# 🎮 Steam Game Recommendation Engine — Semantic Embeddings

A content-based recommender that finds similar Steam games using **sentence embeddings** of their store descriptions (as opposed to tag-based similarity).

**Pipeline:**
1. Load game descriptions and metadata
2. Embed short/detailed descriptions with `sentence-transformers` (`all-MiniLM-L6-v2`)
3. Compute cosine similarity between embeddings
4. Validate on hand-picked game clusters (Soulslike, cozy, FPS, horror, etc.)
5. Build a reusable "top-K similar games" function
6. Embed descriptions for the entire catalog

**Data files required** (place in the same directory as this notebook):
- `steam_games_clean.csv`
- `steam_descriptions_clean.csv`


## 1. Setup

Import core libraries.

In [2]:
import numpy as np
import pandas as pd

## 2. Load Descriptions

Load the cleaned Steam descriptions dataset and take a first look.

In [3]:
descriptions_df = pd.read_csv("steam_descriptions_clean.csv", engine = 'python')

print(descriptions_df.shape)
print(descriptions_df.columns)

descriptions_df.head()

(27334, 3)
Index(['steam_appid', 'short_description', 'detailed_description'], dtype='str')


,steam_appid,short_description,detailed_description
0,10,Play the world's number 1 online action game. ...,Play the world's number 1 online action game. ...
1,20,One of the most popular online action games of...,One of the most popular online action games of...
2,30,Enlist in an intense brand of Axis vs. Allied ...,Enlist in an intense brand of Axis vs. Allied ...
3,40,Enjoy fast-paced multiplayer gaming with Death...,Enjoy fast-paced multiplayer gaming with Death...
4,50,Return to the Black Mesa Research Facility as ...,Return to the Black Mesa Research Facility as ...


Check for missing/empty short descriptions and preview a few examples:

In [4]:
print("Missing short descriptions:", descriptions_df["short_description"].isna().sum())
print("Empty short descriptions:",
      (descriptions_df["short_description"].fillna("").str.strip() == "").sum())

print("\nExample descriptions:\n")

for text in descriptions_df["short_description"].dropna().head(10):
    print(text)
    print("-" * 100)

Missing short descriptions: 0
Empty short descriptions: 0

Example descriptions:

Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.
----------------------------------------------------------------------------------------------------
One of the most popular online action games of all time, Team Fortress Classic features over nine character classes -- from Medic to Spy to Demolition Man -- enlisted in a unique style of online team warfare. Each character class possesses unique weapons, items, and abilities, as teams compete online in a variety of game play modes.
----------------------------------------------------------------------------------------------------
Enlist in an intense brand of Axis vs. Allied teamplay set in the

## 3. Install & Load the Embedding Model

Use `sentence-transformers` with the lightweight `all-MiniLM-L6-v2` model to turn text descriptions into dense vectors.

In [5]:
!pip install -q sentence-transformers


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [6]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer(
    "all-MiniLM-L6-v2"
)

/Users/rayyan/Code/Projects/SteamMCP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9900.16it/s]


## 4. Load Game Metadata

In [7]:
games_df = pd.read_csv("steam_games_clean.csv")

print(games_df.shape)
print(games_df[["appid", "name"]].head())

(27075, 18)
   appid                       name
0     10             Counter-Strike
1     20      Team Fortress Classic
2     30              Day of Defeat
3     40         Deathmatch Classic
4     50  Half-Life: Opposing Force


### 4.1 Game Search Helper

Simple substring search to find a game's `appid` by name.

In [8]:
def search_game(query, n=10):
    results = games_df[
        games_df["name"].str.contains(query, case=False, na=False)
    ][["appid", "name"]]

    return results.head(n)

Try it on a few titles:

In [9]:
search_game("Hollow Knight")

,appid,name
5554,367520,Hollow Knight


In [10]:
search_game("Ori and the Blind Forest")

,appid,name
6281,387290,Ori and the Blind Forest: Definitive Edition


In [11]:
search_game("Dead Cells")

,appid,name
13204,588650,Dead Cells


In [12]:
search_game("Dark Souls")

,appid,name
1728,236430,DARK SOULS™ II
4269,335300,DARK SOULS™ II: Scholar of the First Sin
5817,374320,DARK SOULS™ III
12543,570940,DARK SOULS™: REMASTERED


In [13]:
search_game("Counter-Strike")

,appid,name
0,10,Counter-Strike
7,80,Counter-Strike: Condition Zero
10,240,Counter-Strike: Source
25,730,Counter-Strike: Global Offensive
2502,273110,Counter-Strike Nexon: Zombies


## 5. First Similarity Test

Pick a handful of games, pull their short descriptions, and embed them to sanity-check the approach.

In [14]:
test_appids = [367520, 387290, 588650, 10]
test_games = games_df[
    games_df["appid"].isin(test_appids)
][["appid", "name"]].merge(
    descriptions_df[["steam_appid", "short_description"]],
    left_on="appid",
    right_on="steam_appid",
    how="left"
)

test_games[["name", "short_description"]]

,name,short_description
0,Counter-Strike,Play the world's number 1 online action game. ...
1,Hollow Knight,Forge your own path in Hollow Knight! An epic ...
2,Ori and the Blind Forest: Definitive Edition,“Ori and the Blind Forest” tells the tale of a...
3,Dead Cells,"Dead Cells is a rogue-lite, metroidvania inspi..."


In [15]:
test_embeddings = embedding_model.encode(
    test_games["short_description"].tolist(),
    normalize_embeddings=True
)

print("Embedding shape:", test_embeddings.shape)

Embedding shape: (4, 384)


Compute pairwise cosine similarity (dot product, since embeddings are normalized):

In [16]:
import numpy as np
import pandas as pd

similarity_matrix = np.dot(test_embeddings, test_embeddings.T)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=test_games["name"].values,
    columns=test_games["name"].values
)

similarity_df.round(4)

,Counter-Strike,Hollow Knight,Ori and the Blind Forest: Definitive Edition,Dead Cells
Counter-Strike,1.0000,0.2218,0.1211,0.2123
Hollow Knight,0.2218,1.0000,0.2407,0.3142
Ori and the Blind Forest: Definitive Edition,0.1211,0.2407,1.0000,0.1907
Dead Cells,0.2123,0.3142,0.1907,1.0000


## 6. Expanded Test — Soulslike Cluster

Add a few more Souls-like/action titles and re-check similarity.

In [17]:
dark_souls_appids = [236430, 374320, 570940,367520, 387290, 588650, 10]

dark_souls_games = games_df[
    games_df["appid"].isin(dark_souls_appids)
][["appid", "name"]].merge(
    descriptions_df[["steam_appid", "short_description"]],
    left_on="appid",
    right_on="steam_appid",
    how="left"
)

dark_souls_games[["name", "short_description"]]

,name,short_description
0,Counter-Strike,Play the world's number 1 online action game. ...
1,DARK SOULS™ II,"Developed by FROM SOFTWARE, DARK SOULS™ II is ..."
2,Hollow Knight,Forge your own path in Hollow Knight! An epic ...
3,DARK SOULS™ III,Dark Souls continues to push the boundaries wi...
4,Ori and the Blind Forest: Definitive Edition,“Ori and the Blind Forest” tells the tale of a...
5,DARK SOULS™: REMASTERED,"Then, there was fire. Re-experience the critic..."
6,Dead Cells,"Dead Cells is a rogue-lite, metroidvania inspi..."


In [18]:
dark_souls_embeddings = embedding_model.encode(
    dark_souls_games["short_description"].tolist(),
    normalize_embeddings=True
)

dark_souls_similarity = np.dot(
    dark_souls_embeddings,
    dark_souls_embeddings.T
)

pd.DataFrame(
    dark_souls_similarity,
    index=dark_souls_games["name"].values,
    columns=dark_souls_games["name"].values
).round(4)

,Counter-Strike,DARK SOULS™ II,Hollow Knight,DARK SOULS™ III,Ori and the Blind Forest: Definitive Edition,DARK SOULS™: REMASTERED,Dead Cells
Counter-Strike,1.0000,0.2107,0.2218,0.1888,0.1211,0.2010,0.2123
DARK SOULS™ II,0.2107,1.0000,0.2599,0.6379,0.1859,0.3674,0.1998
Hollow Knight,0.2218,0.2599,1.0000,0.2936,0.2407,0.1795,0.3142
DARK SOULS™ III,0.1888,0.6379,0.2936,1.0000,0.2902,0.3562,0.2631
Ori and the Blind Forest: Definitive Edition,0.1211,0.1859,0.2407,0.2902,1.0000,0.2853,0.1907
DARK SOULS™: REMASTERED,0.2010,0.3674,0.1795,0.3562,0.2853,1.0000,0.2189
Dead Cells,0.2123,0.1998,0.3142,0.2631,0.1907,0.2189,1.0000


Print out the raw descriptions for a manual/qualitative check:

In [19]:
for _, row in dark_souls_games.iterrows():
    print("=" * 100)
    print(row["name"])
    print("=" * 100)
    print(row["short_description"])
    print()

Counter-Strike
Play the world's number 1 online action game. Engage in an incredibly realistic brand of terrorist warfare in this wildly popular team-based game. Ally with teammates to complete strategic missions. Take out enemy sites. Rescue hostages. Your role affects your team's success. Your team's success affects your role.

DARK SOULS™ II
Developed by FROM SOFTWARE, DARK SOULS™ II is the highly anticipated sequel to the gruelling 2011 breakout hit Dark Souls. The unique old-school action RPG experience captivated imaginations of gamers worldwide with incredible challenge and intense emotional reward.

Hollow Knight
Forge your own path in Hollow Knight! An epic action adventure through a vast ruined kingdom of insects and heroes. Explore twisting caverns, battle tainted creatures and befriend bizarre bugs, all in a classic, hand-drawn 2D style.

DARK SOULS™ III
Dark Souls continues to push the boundaries with the latest, ambitious chapter in the critically-acclaimed and genre-defi

## 7. Build a Broader Comparison Set

Search for a wider variety of games spanning several genres (soulslike, cozy, FPS, horror, atmospheric platformers, puzzle/sandbox) to stress-test the embeddings.

In [20]:
games_to_search = [
    "Dark Souls III",
    "Bloodborne",
    "The Surge",
    "Nioh",
    "Stardew Valley",
    "Slime Rancher",
    "A Hat in Time",
    "Unpacking",
    "DOOM",
    "Wolfenstein: The New Order",
    "SUPERHOT",
    "Killing Floor 2",
    "Resident Evil 7: Biohazard",
    "Outlast",
    "Amnesia: The Dark Descent",
    "Alien: Isolation",
    "Limbo",
    "Inside",
    "Portal 2",
    "Garry's Mod"
]
for game in games_to_search:
    print("\n" + "=" * 100)
    print(f"MATCHES FOR: {game}")
    print("=" * 100)

    print(
        search_game(game)
        .to_string(index=False)
    )


MATCHES FOR: Dark Souls III
Empty DataFrame
Columns: [appid, name]
Index: []

MATCHES FOR: Bloodborne
Empty DataFrame
Columns: [appid, name]
Index: []

MATCHES FOR: The Surge
 appid      name
378540 The Surge

MATCHES FOR: Nioh
 appid                                         name
485510 Nioh: Complete Edition / 仁王 Complete Edition

MATCHES FOR: Stardew Valley
 appid           name
413150 Stardew Valley

MATCHES FOR: Slime Rancher
 appid          name
433340 Slime Rancher

MATCHES FOR: A Hat in Time
 appid          name
253230 A Hat in Time

MATCHES FOR: Unpacking
Empty DataFrame
Columns: [appid, name]
Index: []

MATCHES FOR: DOOM
 appid                              name
  2280                     Ultimate Doom
  2290                        Final DOOM
  2300                           DOOM II
  9050                            DOOM 3
  9160         Master Levels for Doom II
 41400                        Doom Rails
208200               Doom 3: BFG Edition
264260 Global Outbreak: Doomsday E

### 7.1 Similarity Across Genre Clusters

Using the confirmed appids from the search above, group games into genre clusters (soulslike, cozy, FPS, horror, atmospheric platformers, puzzle/sandbox) and compute the full similarity matrix on **short descriptions**.

In [21]:
test_appids = [
    # Soulslike
    374320,  # DARK SOULS III
    378540,  # The Surge
    485510,  # Nioh

    # Cozy / colourful
    413150,  # Stardew Valley
    433340,  # Slime Rancher
    253230,  # A Hat in Time

    # FPS / action
    201810,  # Wolfenstein: The New Order
    322500,  # SUPERHOT
    232090,  # Killing Floor 2
    2300,    # DOOM II

    # Horror
    238320,  # Outlast
    57300,   # Amnesia: The Dark Descent
    214490,  # Alien: Isolation

    # Atmospheric platformers
    48000,   # LIMBO
    304430,  # INSIDE

    # Puzzle / sandbox
    620,     # Portal 2
    4000     # Garry's Mod
]

comparison_games = games_df[
    games_df["appid"].isin(test_appids)
][["appid", "name"]].merge(
    descriptions_df[["steam_appid", "short_description"]],
    left_on="appid",
    right_on="steam_appid",
    how="left"
)

comparison_embeddings = embedding_model.encode(
    comparison_games["short_description"].tolist(),
    normalize_embeddings=True
)

similarity_matrix = np.dot(
    comparison_embeddings,
    comparison_embeddings.T
)

similarity_df = pd.DataFrame(
    similarity_matrix,
    index=comparison_games["name"].values,
    columns=comparison_games["name"].values
)

similarity_df.round(4)

,Portal 2,DOOM II,Garry's Mod,LIMBO,Amnesia: The Dark Descent,Wolfenstein: The New Order,Alien: Isolation,Killing Floor 2,Outlast,A Hat in Time,INSIDE,SUPERHOT,DARK SOULS™ III,The Surge,Stardew Valley,Slime Rancher,Nioh: Complete Edition / 仁王 Complete Edition
Portal 2,1.0000,0.1128,0.3328,-0.0212,0.1323,0.1239,0.0706,0.3519,0.2158,0.1997,0.0741,0.1317,0.1171,0.1365,0.0789,-0.0514,0.1064
DOOM II,0.1128,1.0000,0.0993,0.1361,0.2375,0.0995,0.2719,0.2506,0.3894,0.1743,0.1082,0.0020,0.3908,0.1577,0.1585,0.2162,0.1764
Garry's Mod,0.3328,0.0993,1.0000,0.0046,0.1933,0.2919,0.0768,0.2225,0.1577,0.2580,0.1482,0.3182,0.1287,0.1160,0.1564,-0.0217,0.1984
LIMBO,-0.0212,0.1361,0.0046,1.0000,0.1059,0.0089,0.1081,0.0278,0.0863,0.1227,0.3357,0.0177,0.1237,0.1964,0.0757,0.0413,0.1880
Amnesia: The Dark Descent,0.1323,0.2375,0.1933,0.1059,1.0000,0.3950,0.4092,0.3161,0.3958,0.1909,0.2438,0.1712,0.4994,0.1828,0.2372,0.0191,0.3816
Wolfenstein: The New Order,0.1239,0.0995,0.2919,0.0089,0.3950,1.0000,0.1422,0.2430,0.2830,0.1373,0.1512,0.3123,0.3918,0.0780,0.1249,0.1066,0.3698
Alien: Isolation,0.0706,0.2719,0.0768,0.1081,0.4092,0.1422,1.0000,0.2114,0.4164,0.0951,0.2129,-0.0160,0.3170,0.1437,0.0797,0.1013,0.1622
Killing Floor 2,0.3519,0.2506,0.2225,0.0278,0.3161,0.2430,0.2114,1.0000,0.2794,0.1363,0.1251,0.2465,0.2381,0.1795,0.1804,0.0207,0.4057
Outlast,0.2158,0.3894,0.1577,0.0863,0.3958,0.2830,0.4164,0.2794,1.0000,0.1724,0.1335,0.1759,0.4184,0.2162,0.1901,0.1034,0.3536
A Hat in Time,0.1997,0.1743,0.2580,0.1227,0.1909,0.1373,0.0951,0.1363,0.1724,1.0000,0.1336,0.2477,0.2332,0.1825,0.1131,0.1227,0.1236


## 8. Repeat with Detailed Descriptions

Short descriptions are terse marketing blurbs — try the same comparison using the full `detailed_description` field (HTML-stripped) to see if it improves separation between clusters.

In [22]:
from bs4 import BeautifulSoup

# Same games as before
test_appids = [
    # Soulslike
    374320,  # DARK SOULS III
    378540,  # The Surge
    485510,  # Nioh

    # Cozy / colourful
    413150,  # Stardew Valley
    433340,  # Slime Rancher
    253230,  # A Hat in Time

    # FPS / action
    201810,  # Wolfenstein: The New Order
    322500,  # SUPERHOT
    232090,  # Killing Floor 2
    2300,    # DOOM II

    # Horror
    238320,  # Outlast
    57300,   # Amnesia: The Dark Descent
    214490,  # Alien: Isolation

    # Atmospheric platformers
    48000,   # LIMBO
    304430,  # INSIDE

    # Puzzle / sandbox
    620,     # Portal 2
    4000     # Garry's Mod
]


# Get detailed descriptions
comparison_games_long = games_df[
    games_df["appid"].isin(test_appids)
][["appid", "name"]].merge(
    descriptions_df[["steam_appid", "detailed_description"]],
    left_on="appid",
    right_on="steam_appid",
    how="left"
)

# Remove HTML
comparison_games_long["clean_description"] = (
    comparison_games_long["detailed_description"]
    .apply(lambda text: BeautifulSoup(str(text), "html.parser").get_text(" "))
)

# Create embeddings
long_embeddings = embedding_model.encode(
    comparison_games_long["clean_description"].tolist(),
    normalize_embeddings=True
)

# Similarity matrix
long_similarity = np.dot(long_embeddings, long_embeddings.T)

long_similarity_df = pd.DataFrame(
    long_similarity,
    index=comparison_games_long["name"].values,
    columns=comparison_games_long["name"].values
)

long_similarity_df.round(4)

ModuleNotFoundError: No module named 'bs4'

## 9. Top-K Similar Games

A reusable function to fetch the top-K most similar games to a query game, given a set of embeddings. Compare results from short-description embeddings vs. detailed-description embeddings side by side.

In [23]:
import numpy as np
import pandas as pd

# ============================================================
# CHANGE THIS TO THE GAME YOU WANT TO TEST
# ============================================================

query_game = "DARK SOULS™ III"


# ============================================================
# FUNCTION: GET TOP 10 SIMILAR GAMES
# ============================================================

def get_top_similar_games(query_game, embeddings, games_dataframe, top_k=10):

    # Find the query game's position
    matches = games_dataframe[
        games_dataframe["name"].str.lower() == query_game.lower()
    ]

    if matches.empty:
        print(f"Game not found: {query_game}")
        return None

    query_index = matches.index[0]

    # IMPORTANT:
    # Resetting index ensures dataframe position matches embedding position
    query_position = games_dataframe.index.get_loc(query_index)

    # Embeddings are already normalized,
    # so dot product = cosine similarity
    similarities = np.dot(
        embeddings,
        embeddings[query_position]
    )

    # Sort from highest similarity to lowest
    top_indices = np.argsort(similarities)[::-1]

    # Remove the query game itself
    top_indices = top_indices[top_indices != query_position]

    # Take top K
    top_indices = top_indices[:top_k]

    results = games_dataframe.iloc[top_indices][
        ["appid", "name"]
    ].copy()

    results["similarity"] = similarities[top_indices]

    results.insert(
        0,
        "rank",
        range(1, len(results) + 1)
    )

    return results


# ============================================================
# SHORT DESCRIPTION RESULTS
# ============================================================

print("=" * 80)
print(f"TOP 10 — SHORT DESCRIPTION EMBEDDINGS")
print(f"QUERY: {query_game}")
print("=" * 80)

short_results = get_top_similar_games(
    query_game=query_game,
    embeddings=comparison_embeddings,
    games_dataframe=comparison_games,
    top_k=10
)

display(short_results)


# ============================================================
# DETAILED DESCRIPTION RESULTS
# ============================================================

print("=" * 80)
print(f"TOP 10 — DETAILED DESCRIPTION EMBEDDINGS")
print(f"QUERY: {query_game}")
print("=" * 80)

detailed_results = get_top_similar_games(
    query_game=query_game,
    embeddings=long_embeddings,
    games_dataframe=comparison_games_long,
    top_k=10
)

display(detailed_results)

TOP 10 — SHORT DESCRIPTION EMBEDDINGS
QUERY: DARK SOULS™ III


,rank,appid,name,similarity
4,1,57300,Amnesia: The Dark Descent,0.499392
8,2,238320,Outlast,0.418418
5,3,201810,Wolfenstein: The New Order,0.391799
1,4,2300,DOOM II,0.390834
16,5,485510,Nioh: Complete Edition / 仁王 Complete Edition,0.332297
6,6,214490,Alien: Isolation,0.317025
10,7,304430,INSIDE,0.245104
7,8,232090,Killing Floor 2,0.238101
9,9,253230,A Hat in Time,0.233202
14,10,413150,Stardew Valley,0.136265


TOP 10 — DETAILED DESCRIPTION EMBEDDINGS
QUERY: DARK SOULS™ III


NameError: name 'long_embeddings' is not defined

## 10. Scale Up — Embed the Full Catalog

Generate short-description embeddings for every game in the dataset (not just the hand-picked test set), so the recommender can operate catalog-wide.

In [24]:
import numpy as np

# Create embeddings for every game's short description
short_embeddings = embedding_model.encode(
    descriptions_df["short_description"].tolist(),
    batch_size=64,
    show_progress_bar=True,
    normalize_embeddings=True
)

# Convert to numpy array
short_embeddings = np.array(short_embeddings)

print("Embedding shape:", short_embeddings.shape)
print("Number of games:", len(descriptions_df))
print("Embedding dimension:", short_embeddings.shape[1])

Batches: 100%|██████████| 428/428 [00:46<00:00,  9.29it/s]

Embedding shape: (27334, 384)
Number of games: 27334
Embedding dimension: 384


In [25]:
import numpy as np

np.save("short_description_embeddings.npy", short_embeddings)

print("Saved successfully!")

Saved successfully!


In [26]:
np.save(
    "short_description_appids.npy",
    descriptions_df["steam_appid"].values
)

print("App IDs saved successfully!")

App IDs saved successfully!


In [27]:
print(type(short_embeddings))
print(short_embeddings.shape)

<class 'numpy.ndarray'>
(27334, 384)


In [28]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# Load data
games_df = pd.read_csv("steam_games_clean.csv")
descriptions_df = pd.read_csv("steam_descriptions_clean.csv")

# Load embeddings and their corresponding app IDs
short_embeddings = np.load("short_description_embeddings.npy")
short_description_appids = np.load("short_description_appids.npy")

# Load the same model used to create embeddings
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

print("Everything loaded!")
print("Embeddings:", short_embeddings.shape)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7719.35it/s]


Everything loaded!
Embeddings: (27334, 384)


In [29]:
def semantic_search(query, top_k=10):

    # Convert user query into an embedding
    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    # Cosine similarity
    similarities = np.dot(short_embeddings, query_embedding)

    # Get top results
    top_indices = np.argsort(similarities)[::-1][:top_k]

    # Get corresponding app IDs and scores
    results = pd.DataFrame({
        "appid": short_description_appids[top_indices],
        "semantic_score": similarities[top_indices]
    })

    # Add game names
    results = results.merge(
        games_df[["appid", "name"]],
        on="appid",
        how="left"
    )

    return results[["name", "appid", "semantic_score"]]

In [37]:
semantic_search(
    "Romance Action Fantasy"
)

,name,appid,semantic_score
0,Anicon - Animal Complex - Cat's Path,502120,0.654500
1,Anicon - Animal Complex - Sheep's Path,611390,0.654500
2,The Beard in the Mirror,385840,0.642045
3,Pinewood Island,710950,0.606763
4,Changeling,1010240,0.598266
5,Crystalline,616250,0.596107
6,Ayni Fairyland,885790,0.588547
7,Nightshade／百花百狼,512180,0.575913
8,Queen Of Thieves,524200,0.566387
9,In Celebration of Violence,509570,0.556904


In [38]:
def semantic_search_with_description(query, top_k=10):

    query_embedding = embedding_model.encode(
        query,
        normalize_embeddings=True
    )

    similarities = np.dot(short_embeddings, query_embedding)

    top_indices = np.argsort(similarities)[::-1][:top_k]

    results = pd.DataFrame({
        "appid": short_description_appids[top_indices],
        "semantic_score": similarities[top_indices]
    })

    results = results.merge(
        games_df[["appid", "name"]],
        on="appid",
        how="left"
    )

    results = results.merge(
        descriptions_df[["steam_appid", "short_description"]],
        left_on="appid",
        right_on="steam_appid",
        how="left"
    )

    return results[
        ["name", "appid", "semantic_score", "short_description"]
    ]

In [39]:
semantic_search_with_description(
    "I want a difficult dark fantasy action game with challenging combat"
)

,name,appid,semantic_score,short_description
0,Mana Spark,630720,0.688507,A challenging action RPG with deep souls-like ...
1,Battle Motion,1000480,0.677876,Single-player action game with massive fantasy...
2,Chronicles of a Dark Lord: Rhapsody Clash,418310,0.660941,Experience the battles of the Chronicles of a ...
3,Battle for Mountain Throne,759840,0.645117,Battle for Mountain Throne - virtual action ga...
4,Kronos,562090,0.643466,"A unique Action RPG game with lots of combats,..."
5,DarkMaus,406130,0.638656,DarkMaus is an indie action RPG with challengi...
6,Ys: The Oath in Felghana,207320,0.635459,An Action RPG driven by fast-paced combat and ...
7,Inquisitor,241620,0.634458,Action-oriented combat with a deep and involvi...
8,Medieval Story,543740,0.632680,An action adventure game taking place in a med...
9,Glaive,505680,0.631727,Battle monsters of the darkness with a kinetic...


In [40]:
query = "I want a difficult dark fantasy action game with challenging combat"

query_embedding = embedding_model.encode(
    query,
    normalize_embeddings=True
)

similarities = np.dot(short_embeddings, query_embedding)

# Create results for every game
all_results = pd.DataFrame({
    "appid": short_description_appids,
    "semantic_score": similarities
})

all_results = all_results.merge(
    games_df[["appid", "name"]],
    on="appid",
    how="left"
)

all_results = all_results.sort_values(
    "semantic_score",
    ascending=False
).reset_index(drop=True)

all_results["rank"] = all_results.index + 1

# Check known relevant games
targets = [
    "DARK SOULS™ III",
    "Nioh",
    "The Surge"
]

all_results[
    all_results["name"].str.contains(
        "DARK SOULS|Nioh|The Surge",
        case=False,
        na=False
    )
][["rank", "name", "semantic_score"]]

,rank,name,semantic_score
640,641,DARK SOULS™ II: Scholar of the First Sin,0.465746
1045,1046,DARK SOULS™ II,0.442116
1677,1678,DARK SOULS™ III,0.418659
1758,1759,Nioh: Complete Edition / 仁王 Complete Edition,0.415774
5848,5849,DARK SOULS™: REMASTERED,0.334673
26120,26121,The Surge,0.042535
